In [1]:
import pyarrow.parquet as pq
table = pq.read_table("dataset/train-00000-of-00001.parquet")
df = table.to_pandas()
df.columns

Index(['instruction', 'input', 'output', 'source', 'persona', 'perspective',
       'category'],
      dtype='str')

In [2]:
import pandas as pd

SYSTEM_PROMPT = (
    "You are CyberCTRL, an expert cybersecurity assistant. "
    "Provide technically correct, clear, and complete answers."
)

def create_messages(row):
    user_prompt = row["instruction"].strip()
    if pd.notna(row["input"]) and row["input"].strip():
        user_prompt += "\n\n" + row["input"].strip()
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
        {
            "role": "assistant",
            "content": row["output"].strip(),
        },
    ]

In [3]:
df["messages"] = df.apply(create_messages, axis=1)
df['messages'][0]

[{'role': 'system',
  'content': 'You are CyberCTRL, an expert cybersecurity assistant. Provide technically correct, clear, and complete answers.'},
 {'role': 'user',
  'content': 'Can you explain CVE-2026-39528 in simple terms?'},
 {'role': 'assistant',
  'content': 'CVE-2026-39528 is a cybersecurity vulnerability with a CVSS score of 5.3 (MEDIUM). In simple terms, it means Missing Authorization vulnerability in WP Delicious WP Delicious delicious-recipes allows Exploiting Incorrectly Configured Access Control Security Levels.This issue affects WP Delicious: from n/a through <= 1.9.5. This matters because it affects various systems and could allow attackers to compromise systems if left unpatched.'}]

In [4]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["persona"],
)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

In [5]:
from unsloth import FastLanguageModel
import os

os.environ["HF_HOME"] = "./models"

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

/Users/armaanjagirdar/Projects/Cyber_LLM/env/lib/python3.12/site-packages/trl/extras/best_of_n_sampler.py:24: FutureWarning: `BestOfNSampler` is deprecated and will be removed in TRL 0.25.
  class BestOfNSampler:


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Unsloth: Loading unsloth/Qwen2.5-3B-Instruct via mlx-lm (runtime 4-bit affine quantization)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

[INFO] Quantized model with 5.660 bits per weight.
Unsloth: Quantized text model to 4-bit affine.


In [7]:
def formatting(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_ds = train_ds.map(formatting)
val_ds = val_ds.map(formatting)

Map:   0%|          | 0/156411 [00:00<?, ? examples/s]

Map:   0%|          | 0/17379 [00:00<?, ? examples/s]

In [8]:
train_ds[0]["text"]

"<|im_start|>system\nYou are CyberCTRL, an expert cybersecurity assistant. Provide technically correct, clear, and complete answers.<|im_end|>\n<|im_start|>user\nWhat immediate steps should a security team take for ?<|im_end|>\n<|im_start|>assistant\nFor  (CVSS N/A, INFO), a security team should:\n1. Identify all assets running affected software: various systems\n2. Apply vendor patches immediately if available\n3. Implement virtual patching / WAF rules if patch cannot be applied\n4. Monitor for active exploitation attempts\n5. Review the following references for detailed guidance: \nDescription of vulnerability: V-Webmail 1.6.4 - 'pear_dir' Remote File Inclusion<|im_end|>\n"

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: LoRA applied — 29,933,568 trainable params (3.86% of 774,860,800 total)


In [10]:
from trl import SFTTrainer, SFTConfig
import torch

training_args = SFTConfig(
    output_dir="CyberCTRL",
    max_seq_length=max_seq_length,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    packing=True,
    report_to="none",
)

/Users/armaanjagirdar/Projects/Cyber_LLM/env/lib/python3.12/site-packages/unsloth/__init__.py:1368: RuntimeWarning: Unsloth MLX: accepting but not applying unsupported TrainingArguments kwargs: bf16, eval_strategy, save_strategy. These options are not implemented by MLXTrainer yet.
  super().__init__(*args, **kwargs)


In [11]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    args=training_args,
)

Unsloth: packing=True is not yet supported on MLX. Falling back to packing=False (standard padding).


In [17]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

RuntimeError: [metal::malloc] Resource limit (499000) exceeded.

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("CyberCTRL_Qwen_3B")
tokenizer.save_pretrained("CyberCTRL_Qwen_3B")